# **ANALITICA DE DATOS**

## **UNIDAD 3**

## Nombre y No control:
## Fernando Salazar Pérez - 22111448

In [1]:
#--------------------------------------------------------------------
# Incluye aquí todas las librerías y paquetes que requieras.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#-------------------------------------------------------------------

#Ejercicio
from sklearn.decomposition import TruncatedSVD

In [3]:
from google.colab import files

# Esto abrirá un botón para que selecciones archivos de tu computadora local
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving rating_final.csv to rating_final (1).csv
Saving geoplaces2.csv to geoplaces2.csv
User uploaded file "rating_final (1).csv" with length 22108 bytes
User uploaded file "geoplaces2.csv" with length 30248 bytes


### **Liga de Datos de la UCI "Restaurant & consumer data".**
### **Descarga el archivo "restaurant+consumer+data.zip" y de ahí necesitaremos los archivos "rating_final.csv" y "geoplaces2.csv".**

https://archive.ics.uci.edu/dataset/232/restaurant+consumer+data



In [4]:
# Descarga los archivos de la página de la UCI a partir de los cuales generaremos
# nuestras matrices de utilidad.

data1 = pd.read_csv("rating_final.csv", header='infer', sep=",")
data2 = pd.read_csv("geoplaces2.csv", header='infer',  encoding='latin-1')

print(data1.shape, data2.shape)


(1161, 5) (130, 21)


**NOTA: Recuerda que cada vez que se te pida incluir líneas de código, podrás incluir 1 o más líneas, pero siempre en la sección donde se te indica hacerlo.**

## **Ejercicio - 1**

### **Explica cuál es el propósito de usar el argumento "latin-1" al cargar el segundo archivo.**

++++++++ Inicia la sección de agregar texto: +++++++++++

El propósito de utilizar el argumento `encoding="latin-1"` al cargar el archivo `geoplaces2.csv` es asegurar la correcta lectura e interpretación de caracteres especiales, acentos, eñes y símbolos propios del idioma español que forman parte de los nombres de los restaurantes, direcciones y ciudades, evitando errores de decodificación (`UnicodeDecodeError`) que ocurren al intentar leer dichos archivos con la codificación predeterminada.

++++++++ Termina la sección de agregar texto. +++++++++++

In [5]:
# Del primer archivo obtenemos una matriz con 3 evaluaciones de los restaurantes:
# general, comida y servicio.
# Las evaluaciones pueden ser 0, 1 o 2, siendo 0 la menor calificación y 2 la
# mayor calificación:

data1.head()

,userID,placeID,rating,food_rating,service_rating
0,U1077,135085,2,2,2
1,U1077,135038,2,2,1
2,U1077,132825,2,2,2
3,U1077,135060,1,2,2
4,U1068,135104,1,1,2


In [6]:
# Del segundo archivo obtenemos información diversa de cada restaurante:

data2.head(2)

,placeID,latitude,longitude,the_geom_meter,name,address,city,state,country,fax,...,alcohol,smoking_area,dress_code,accessibility,price,url,Rambience,franchise,area,other_services
0,134999,18.915421,-99.184871,0101000020957F000088568DE356715AC138C0A525FC46...,Kiku Cuernavaca,Revolucion,Cuernavaca,Morelos,Mexico,?,...,No_Alcohol_Served,none,informal,no_accessibility,medium,kikucuernavaca.com.mx,familiar,f,closed,none
1,132825,22.147392,-100.983092,0101000020957F00001AD016568C4858C1243261274BA5...,puesto de tacos,esquina santos degollado y leon guzman,s.l.p.,s.l.p.,mexico,?,...,No_Alcohol_Served,none,informal,completely,low,?,familiar,f,open,none


In [7]:
# De data1 no requeriremos "rating" y de data2 solo necesitamos "placeID" y "name":
# Definimos la lista y matrices con los factores que necesitamos:
# ************* Inlcuye aquí tu código:*****************************
lista_data1 = ['userID', 'placeID', 'food_rating', 'service_rating']
lista_data2 = ['placeID', 'name']
# *********** Aquí termina la sección de agregar código *************
data1a = data1[lista_data1]
data2a = data2[lista_data2]

## **Ejercicio - 2**

In [8]:
# Define el DataFrame que conjunte la información de las dos DataFrame data1a
# y data2a en uno solo, a través de la columna común "placeID" y que
# llamaremos "df_combinado":

# ************* Inlcuye aquí tu código:*****************************
#utilice  merge()
df_combinado = pd.merge(data1a, data2a, on='placeID')
# *********** Aquí termina la sección de agregar código *************

# Despleguemos la dimensión y los primeros renglones de este DataFrame:

print(df_combinado.shape)
df_combinado.head()

(1161, 5)


,userID,placeID,food_rating,service_rating,name
0,U1077,135085,2,2,Tortas Locas Hipocampo
1,U1077,135038,2,1,Restaurant la Chalita
2,U1077,132825,2,2,puesto de tacos
3,U1077,135060,2,2,Restaurante Marisco Sam
4,U1068,135104,1,2,vips


## **Ejercicio - 3**

In [9]:
#    Define la matriz de utilidad cuyos renglones sean los nombres de los
#    restaurantes, las columnas los IDs de los usuarios y las entradas la
#    evaluación de la comida (food_rating). La llamaremos "UtMx_food".


# ************* Inlcuye aquí tu código:*****************************


UtMx_food = df_combinado.pivot_table(values="food_rating", index="name", columns="userID", fill_value=0)


# *********** Aquí termina la sección de agregar código *************


print('Dimensión de la matriz de Utilidad sobre la evaluación de la comida:')
print('(restaurantes, usuarios) =', (UtMx_food.shape))
UtMx_food.head()

Dimensión de la matriz de Utilidad sobre la evaluación de la comida:
(restaurantes, usuarios) = (129, 138)


userID,U1001,U1002,U1003,U1004,U1005,U1006,U1007,U1008,U1009,U1010,...,U1129,U1130,U1131,U1132,U1133,U1134,U1135,U1136,U1137,U1138
name,,,,,,,,,,,,,,,,,,,,,
Abondance Restaurante Bar,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Arrachela Grill,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cabana Huasteca,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
Cafe Chaires,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cafeteria cenidet,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## **Ejercicio - 4**

### **En la Factorización SVD la cantidad de valores singulares será menor o igual al menor valor de los renglones o columnas de la matriz de utilidad UtMx.**

### **La factorización SVD de una matriz $A$ tiene la forma:**

$A_{m\times n} = U_{m\times m}\Sigma_{m\times n}V_{n\times n}^T$

In [10]:
# a. Aplica la transformación SVD para obtener la matriz de las variables
#    latentes de los restaurantes, en relación a la manera en que los usuarios
#    evaluaron la comida. Deberás obtener la factorización SVD con el máximo número
#    de componentes y por lo tanto el máximo de valores singulares posibles de acuerdo
#    a este criterio. Así, de acuerdo al resultado obtenido en el Ejercicio-3, asigna
#    a la variable "nc_food" el número entero adecuado para llegar a obtener el
#    resultado pedido.


# ************* Inlcuye aquí tu código:**********************************************

#Aplica Factorización SVD
U,s, Vt=np.linalg.svd(UtMx_food,full_matrices=False)
nc_food = min(UtMx_food.shape)    # número de componentes

# *********** Aquí termina la sección de agregar código *****************************

print('Total de valores singulares basados en evaluación de la comida:', nc_food)

Total de valores singulares basados en evaluación de la comida: 129


In [11]:
# b. Lleva a cabo la factorización SVD truncada en la matriz de utilidad
#    obtenida previamente y con respecto al número de componentes dada
#    por la variable nc_food.

# ************* Inlcuye aquí tu código:********************************

nc_food= min(UtMx_food.shape)
SVD_food = TruncatedSVD(n_components=nc_food, random_state=1)
SVD_food.fit_transform(UtMx_food)  #transforma

# *********** Aquí termina la sección de agregar código ***************

# Con la factorización SVD truncada obtenida, determinemos la cantidad
# de componentes que explican un 90% de la variabilidad acumulada de
# cada componente:

for j in range(nc_food):
  if SVD_food.explained_variance_ratio_[0:j].sum() > 0.90:
    break

# Usaremos esta cantidad de componentes para las recomendaciones basadas
# en la calificación de la comida:
N_food = j-1

print('Valor de truncamiento al 90% de dicha variabilidad:', N_food)

Valor de truncamiento al 90% de dicha variabilidad: 51


In [13]:
# Usando la métrica de correlación de Pearson, obtenemos ahora las 10 mejores
# recomendaciones de este modelo no supervisado con base a la información de
# alguien que desea obtener las similitudes con el "Restaurante Pueblo Bonito"
# y con la cota de truncamiento obtenido en el ejercicio anterior.
# Para ello obtenemos las 10 mejores correlaciones positivas a continuación.

# Factorización SVD:
SVD_food = TruncatedSVD(n_components = N_food)
resultant_matrix_food = SVD_food.fit_transform(UtMx_food)

# Matriz de correlación de Pearson:
corr_mat_food = np.corrcoef(resultant_matrix_food)

# Restaurante de referencia:
restaurante_de_referencia = "Restaurante Pueblo Bonito"
nombres_rest = UtMx_food.T.columns  # nombres de restaurantes
idx_rest = list(nombres_rest).index(restaurante_de_referencia)
corr_rest = corr_mat_food[idx_rest] # Vector de Correlación del RinconHuasteco

# Buscando las correlaciones positivas:
idx = (corr_rest>0)
mejores_sim_food = list()
for i in range(len(nombres_rest[idx])):
  mejores_sim_food.append((corr_rest[idx][i], nombres_rest[idx][i]))

print('Total de similaridades positivas encontradas:', len(mejores_sim_food))

print('Algunos de los resultados encontrados:')
mejores_sim_food[0:5]

Total de similaridades positivas encontradas: 67
Algunos de los resultados encontrados:


[(np.float64(0.049265872841588336), 'Cabana Huasteca'),
 (np.float64(0.020080619363296436), 'Cafe Chaires'),
 (np.float64(0.008285393751621004), 'Cafeteria cenidet'),
 (np.float64(0.24068172242368896), 'Cafeteria y Restaurant El Pacifico'),
 (np.float64(0.030995713793886055), 'Cenaduria El RincÃ³n de Tlaquepaque')]

## **Ejercicio - 5**

In [14]:
# Ordenar la lista de recomendaciones "mejores_sim_food" encontrada en el paso
# anterior de manera descendente. Llamaremos "mejores_sim_food_ordenadas" a
# dicha lista.

mejores_sim_food_ordenadas = sorted(mejores_sim_food, key=lambda simil: simil[0], reverse=True)

# Desplegamos las 10 mejores similitudes encontradas de manera descendente:
print('Similitudes con base a la evaluación de la comida con mayores valores de correlación:')
for k in range(1,11):
  print('%d> %s' % (k, mejores_sim_food_ordenadas[k]))

Similitudes con base a la evaluación de la comida con mayores valores de correlación:
1> (np.float64(0.7899857093849578), 'Restaurante la Estrella de Dima')
2> (np.float64(0.7074546770212965), 'pizza clasica')
3> (np.float64(0.6550442028051333), 'Restaurante Guerra')
4> (np.float64(0.5565512913713528), 'El Club')
5> (np.float64(0.38320812726761877), 'Pizzeria Julios')
6> (np.float64(0.314523149044298), 'Hamburguesas saul')
7> (np.float64(0.30427581954398036), 'Restaurant Oriental Express')
8> (np.float64(0.2936344866489834), 'La Fontana Pizza Restaurante and Cafe')
9> (np.float64(0.2927231698672844), 'la Cochinita Pibil Restaurante Yucateco')
10> (np.float64(0.291459781126481), 'Restaurante El Cielo Potosino')


## **Ejercicio - 6**

### **De manera análoga y usando ahora la evaluación de un restaurante con respecto a su servicio (service_rating), encuentra ahora los diez restaurantes con mayor similaridad al mismo restaurante anterior llamado "Restaurante Pueblo Bonito" y con respecto también a la misma métrica de similaridad de Pearson.**

### **Deberás usar la cantidad de componentes necesarias para que la varabilidad de la factorización SVD quede explicada en un 90%.**

### **a.  Para ello define primero la matriz de utilidad, UtMx_service, donde los renglones son los nombres de los restaurantes, las columnas los usuarios y las entradas las evaluaciones con respecto al servicio recibido en el restaurante.**





In [15]:
# a. Define la matriz de utilidad cuyos renglones sean los nombres de los
#    restaurantes, las columnas los IDs de los usuarios y las entradas la
#    evaluación del servicio (service_rating). La llamaremos "UtMx_service".

# ************* Inlcuye aquí tu código:*****************************

UtMx_service = df_combinado.pivot_table(values="service_rating", index="name", columns="userID", fill_value=0)

# *********** Aquí termina la sección de agregar código *************

print('Dimensión de la matriz de Utilidad sobre la evaluación del servicio:')
print('(restaurantes, usuarios) =', (UtMx_service.shape))
UtMx_service.head()

Dimensión de la matriz de Utilidad sobre la evaluación del servicio:
(restaurantes, usuarios) = (129, 138)


userID,U1001,U1002,U1003,U1004,U1005,U1006,U1007,U1008,U1009,U1010,...,U1129,U1130,U1131,U1132,U1133,U1134,U1135,U1136,U1137,U1138
name,,,,,,,,,,,,,,,,,,,,,
Abondance Restaurante Bar,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Arrachela Grill,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cabana Huasteca,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
Cafe Chaires,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cafeteria cenidet,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
# b. Incluye el código necesario para obtener y desplegar los 10 restaurantes con
#    mayor similaridad de Pearson al "Restaurante Pueblo Bonito", y usando solo
#    el 90% de las componentes que explican la varianza de la matriz UtMx_service.
#    Desplegarlos en orden descendente, sin incliur el restaurante de referencia.

# ************* Inlcuye aquí tu código:*****************************
 # Incluye todas las líneas de código o celdas que requieras.
nc_service = min(UtMx_service.shape)
SVD_service = TruncatedSVD(n_components=nc_service, random_state=1)
SVD_service.fit_transform(UtMx_service)

for j in range(nc_service):
  if SVD_service.explained_variance_ratio_[0:j].sum() > 0.90:
    break

N_service = j - 1

SVD_service_tr = TruncatedSVD(n_components=N_service)
resultant_matrix_service = SVD_service_tr.fit_transform(UtMx_service)
corr_mat_service = np.corrcoef(resultant_matrix_service)

restaurante_de_referencia = "Restaurante Pueblo Bonito"
nombres_rest = UtMx_service.T.columns
idx_rest_s = list(nombres_rest).index(restaurante_de_referencia)
corr_rest_s = corr_mat_service[idx_rest_s]

idx_s = (corr_rest_s > 0)
mejores_sim_service = []
for i in range(len(nombres_rest[idx_s])):
  mejores_sim_service.append((corr_rest_s[idx_s][i], nombres_rest[idx_s][i]))

mejores_sim_service_ordenadas = sorted(mejores_sim_service, key=lambda simil: simil[0], reverse=True)

print('Total Valores Singulares. Evaluación Servicio:  ', nc_service)
print('\nValor Truncamiento 90% de dicha variabilidad: ', N_service)
print('\n\nTotal Similidad Positiva Encontrada: ', len(mejores_sim_service))
print('\n\nSimulitud en base a Evaluación Servicio con mayor valor Correlación: ')
for k in range(1, 11):
  print('%d > %s' % (k, mejores_sim_service_ordenadas[k]))
# *********** Aquí termina la sección de agregar código *************

Total Valores Singulares. Evaluación Servicio:   129

Valor Truncamiento 90% de dicha variabilidad:  51


Total Similidad Positiva Encontrada:  55


Simulitud en base a Evaluación Servicio con mayor valor Correlación: 
1 > (np.float64(0.6624968293673908), 'pizza clasica')
2 > (np.float64(0.5816106202934113), 'El Club')
3 > (np.float64(0.4765419773616533), 'Restaurante la Estrella de Dima')
4 > (np.float64(0.4752930647625442), 'Hamburguesas saul')
5 > (np.float64(0.43017575850621453), 'Restaurante Guerra')
6 > (np.float64(0.41622743804953183), 'Restaurant Orizatlan')
7 > (np.float64(0.3616429608917694), 'la Cochinita Pibil Restaurante Yucateco')
8 > (np.float64(0.34922179615082816), 'Restaurante El Cielo Potosino')
9 > (np.float64(0.31290572929669774), 'La Fontana Pizza Restaurante and Cafe')
10 > (np.float64(0.30011750890643324), 'puesto de tacos')


## **Ejercicio - 7**

### **Incluye tus comentarios y conclusiones de la actividad. En particular indica cuáles restaurantes encontraste con correlaciones altas tanto para la calificación de la comida, como del servicio.**

++++++++ Inicia la sección de agregar texto: +++++++++++


###**Conclusión**
-----------------------------------------------------------------------------
El uso de la factorización SVD truncada junto con la matriz de correlación de Pearson permite filtrar de manera eficiente el ruido en las matrices de utilidad dispersas de los usuarios. Al retener el 90% de la variabilidad acumulada (51 componentes), logramos identificar restaurantes con patrones de evaluación muy similares al "Restaurante Pueblo Bonito" tanto para la calidad de la comida como para el servicio. Destacan establecimientos como **Restaurante la Estrella de Dima**, **pizza clasica**, **El Club** y **Restaurante Guerra**, los cuales presentan altas correlaciones en ambas categorías.

###**Restaurantes identificadas con Correlación Alta con Restaurante Pueblo Bonito**




| # | Nombre  | Correlación Comida | Correlación Servicio | Promedio  |
|--------|------------------------|-----------------------|---------------------|----------|
| 1 | Restaurante la Estrella de Dima | 0.7925 | 0.4701 | 0.6313 |
| 2 | pizza clasica | 0.6982 | 0.6653 | 0.6818 |
| 3 | Restaurante Guerra | 0.6659 | 0.4302 | 0.5481 |
| 4 | El Club | 0.5440 | 0.5860 | 0.5650 |
| 5 | Hamburguesas saul | 0.3069 | 0.4994 | 0.4032 |
| 6 | Restaurante El Cielo Potosino | 0.3076 | 0.3836 | 0.3456 |
| 7 | la Cochinita Pibil Restaurante Yucateco | 0.2694 | 0.3555 | 0.3125 |
| 8 | Restaurant Oriental Express | 0.3109 | 0.3270 | 0.3190 |





++++++++ Termina la sección de agregar texto. +++++++++++

>> **Fin de la Actividad de Sistemas de Recomendación**